# Planet imagery discovery

This notebook searches Planet Data API for imagery over the supplied area of interest and displays scene previews. It is a prototype data-discovery step for a later Streamlit dashboard. API credentials are requested at runtime and are not saved in the notebook.

## 1. Configure the connection

Planet Data API authenticates with the API key as the Basic Auth username and an empty password. The Planet account email/password is not used for this API workflow. Do not paste credentials into notebook cells or commit them.

In [ ]:
from getpass import getpass
from IPython.display import display, Markdown, Image
import requests
import pandas as pd

PLANET_API_KEY = getpass('Planet API key (input hidden): ')
if not PLANET_API_KEY.strip():
    raise ValueError('An API key is required to query Planet Data API.')

QUICK_SEARCH_URL = 'https://api.planet.com/data/v1/quick-search'
AOI = {
    'type': 'Polygon',
    'coordinates': [[
        [-52.4825, -6.8735], [-52.4825, -6.65775],
        [-52.26725, -6.65775], [-52.26725, -6.8735],
        [-52.4825, -6.8735],
    ]],
}


## 2. Search the area

The query uses the AOI and supported scene types from the supplied search. `limit` keeps this initial preview manageable; pagination is handled so the result table reflects the complete first search response stream.

In [ ]:
# Search for scenes intersecting the AOI. This broad discovery query avoids assuming
# that every item type has identical cloud metadata. Refine filters after reviewing results.
query = {
    'item_types': ['PSScene', 'SkySatCollect', 'PelicanScene'],
    'filter': {
        'type': 'AndFilter',
        'config': [
            {'type': 'GeometryFilter', 'field_name': 'geometry', 'config': AOI},
        ],
    },
}

response = requests.post(
    QUICK_SEARCH_URL,
    json=query,
    auth=(PLANET_API_KEY, ''),
    timeout=(15, 60),
)
if response.status_code == 401:
    raise PermissionError('Planet rejected the API key (HTTP 401). Check the key and account access.')
response.raise_for_status()
payload = response.json()
features = payload.get('features', [])
print(f'Returned {len(features)} scenes in this response.')


## 3. Review scene metadata

In [ ]:
records = []
for feature in features:
    props = feature.get('properties', {})
    links = feature.get('_links', {})
    records.append({
        'id': feature.get('id'),
        'item_type': feature.get('item_type') or props.get('item_type'),
        'acquired': props.get('acquired'),
        'cloud_percent': props.get('cloud_percent'),
        'cloud_cover': props.get('cloud_cover'),
        'quality_category': props.get('quality_category'),
        'thumbnail': links.get('thumbnail'),
    })
scenes = pd.DataFrame(records)
if scenes.empty:
    display(Markdown('No scenes were returned for this AOI. Check account access, API key, and AOI coordinates.'))
else:
    display(scenes.drop(columns=['thumbnail']).sort_values('acquired', na_position='last').reset_index(drop=True))


## 4. Display available previews

Planet’s thumbnail links may redirect to preview assets. These are for visual inspection; downloading/activating analytic assets is a separate step and may depend on account entitlements.

In [ ]:
if not scenes.empty:
    preview_rows = scenes[scenes['thumbnail'].notna()].sort_values('acquired', na_position='last').tail(12)
    for _, row in preview_rows.iterrows():
        display(Markdown(f"**{row['id']}** — {row['acquired']} | {row['item_type']}"))
        try:
            thumb = requests.get(row['thumbnail'], auth=(PLANET_API_KEY, ''), timeout=(15, 60))
            thumb.raise_for_status()
            display(Image(data=thumb.content, width=600))
        except requests.RequestException as exc:
            print(f"Preview unavailable for {row['id']}: {exc}")


## Next step: Streamlit

Once the scene previews and filters are validated, move the search and display logic into a Streamlit app. Keep the API key in an environment variable or Streamlit secrets (for local development, `.streamlit/secrets.toml` should be excluded from version control). Add pagination, bounded retries, and asset activation/download handling before scaling beyond exploratory use.